In [1]:
!pip install underthesea
!pip install vietnam-number
!pip install dask
!git clone https://github.com/stopwords/vietnamese-stopwords
!pip -q install pandarallel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 28.7 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.8/657.8 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.4 MB/s eta 0:00:00
Cloning into 'vietnamese-stopwords'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 95 (delta 3), reused 0 (delta 0), pack-reused 81 (from 1)
Receiving objects: 100% (95/95), 40.25 KiB | 8.05 MiB/s, done.
Resolving deltas: 100% (31/31), done.
  Preparing metadata (setup.py) ... done


In [60]:
import csv
import json
import math
import os
import random
import re
import unicodedata
import zipfile
from collections import Counter, defaultdict
from functools import partial
from multiprocessing import cpu_count
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from underthesea import text_normalize, word_tokenize
from vietnam_number import n2w

import dask.bag as db
from dask import dataframe as dd
from dask.diagnostics import ProgressBar
from dask.multiprocessing import get

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

from pandarallel import pandarallel
from tqdm import tqdm
import matplotlib.pyplot as plt
from tqdm import tqdm
from contextlib import nullcontext

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("iambestfeeder/10000-vietnamese-books")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/10000-vietnamese-books


In [4]:
def load_books(data_directory):
    texts = []
    titles = []
    authors = []

    files = os.listdir(data_directory)

    for file_name in files:
        file_path = os.path.join(data_directory, file_name)

        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read().strip()

            texts.append(content)

            title = file_name.replace('.txt', '').split('-')[:-1]
            author=file_name.replace('.txt', '').split('-')[-1]
            titles.append(title)
            authors.append(author)


    return pd.DataFrame({'Titles': titles, 'Texts': texts,'Authors':authors})

In [5]:
data_directory = '/kaggle/input/10000-vietnamese-books/output'
dataset = load_books(data_directory)
dataset.head()

,Titles,Texts,Authors
0,[Nhà ảo thuật ],Mạc Can\nNhà ảo thuật\nCó một cậu bé muốn học ...,Mạc Can
1,[Ra đi ],Trần Chi Liên\nRa đi\nTặng người đồng cảnh tươ...,Trần Chi Liên
2,[Đường về ],"Vũ Thư Nguyên\nĐường về\nVừa ra khỏi xa lộ 66,...",Vũ Thư Nguyên
3,[Bốn Thằng Buồn ],Phi Va\nBốn Thằng Buồn\n&quot;Bốn người lính b...,Phi Va
4,[Chiếc Lexus và cây Ô liu ],Thomas L. Friedman\nChiếc Lexus và cây Ô liu\n...,Thomas L. Friedman


In [6]:
dataset.shape

(10415, 3)

# Split data

In [7]:
class DataSplitter:
    def __init__(self, train_ratio = 0.7, val_ratio = 0.1, seed = 42, shuffle = True):
        self.train_ratio = train_ratio
        self.val_ratio   = val_ratio
        self.seed        = seed
        self.shuffle     = shuffle
    
    def split(self, data: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        df = data.copy()
        
        if self.shuffle:
            df = df.sample(frac=1, random_state=self.seed).reset_index(drop=True)

        n = len(df)
        n_train = int(n * self.train_ratio)
        n_val   = int(n * self.val_ratio)

        train = df.iloc[:n_train]
        val   = df.iloc[n_train:n_train+n_val]
        test  = df.iloc[n_train+n_val:]
        return train, val, test


In [8]:
splitter = DataSplitter(train_ratio=0.7, val_ratio=0.1, seed=42, shuffle=True)

train_df, val_df, test_df = splitter.split(dataset)

print(len(train_df), len(val_df), len(test_df))

7290 1041 2084


In [9]:
train_df

,Titles,Texts,Authors
0,[Sanh Rớt ],"Hòa Đa\nSanh Rớt\nThời gian gặp lại vợ con, sa...",Hòa Đa
1,[Không Khóc ở California ],Nguyễn Huy Thiệp\nKhông Khóc ở California\nAnh...,Nguyễn Huy Thiệp
2,[Võ Lâm Tình Sử ],Đông Tà\nVõ Lâm Tình Sử\nHồi 1\nHại Thân Vì Bữ...,Đông Tà
3,[Sherlock Holmes trở về ],Sir ARTHUR CONAN DOYLE\nSherlock Holmes trở về...,Sir ARTHUR CONAN DOYLE
4,[Điệu Nam Ai ],Mầu Hoa Khế\nĐiệu Nam Ai\nBắt đầu tới tuổi 13 ...,Mầu Hoa Khế
...,...,...,...
7285,[Con có yêu mẹ không_ ],Peter Carey\nCon có yêu mẹ không?\nLời giới th...,Peter Carey
7286,[Tượng Đài Thương Tiếc ],Mặc Nhiên\nTượng Đài &quot;Thương Tiếc &quot;\...,Mặc Nhiên
7287,[Mất Hút Xuân Thì ],Ấu Tím\nMất Hút Xuân Thì\nTiếng muỗng đĩa lanh...,Ấu Tím
7288,[Mẹ Tôi và bí ẩn tâm linh ],Kim Phương Oanh\nMẹ Tôi và bí ẩn tâm linh\n- M...,Kim Phương Oanh


# Preprocessing

In [10]:
import re, unicodedata, random
import pandas as pd
from typing import List, Iterable, Union

class TextPreprocessor:
    QUOTE_MAP = {
        '“':'"', '”':'"', '„':'"', '‟':'"', '〝':'"', '〞':'"',
        '‘':"'", '’':"'", '‚':"'", '‛':"'", '´':"'", '`':"'"
    }

    DEFAULT_DROP_PATTERNS = [
        r"(?im)^\s*mục\s*lục\s*$",
        r"(?im)^\s*lời\s*cuối.*$",
        r"(?im)^\s*nguồn\s*:.*$",
        r"(?im)^\s*phát\s*hành\s*:.*$",
        r"(?im)^\s*chào mừng.*(dự án|đón đọc).*$",
        r"http[s]?://\S+",
    ]

    def __init__(self,
                 para_token="<PARA>",
                 bos_token="<BOS>",
                 eos_token="<EOS>",
                 min_words=200):
        self.PARA, self.BOS, self.EOS = para_token, bos_token, eos_token
        self.min_words = min_words

        # basic cleans
        self.re_ctrl    = re.compile(r"[\u0000-\u0008\u000B-\u000C\u000E-\u001F]")
        self.re_spaces  = re.compile(r"[ \t]+")
        self.re_manynl  = re.compile(r"\n{3,}")
        self.re_ws_nl   = re.compile(r"[ \t]+(\n)|(\n)[ \t]+")
        self.drop_res   = [re.compile(p) for p in self.DEFAULT_DROP_PATTERNS]

        # punctuation normalization
        self.re_space_before_punct = re.compile(r"\s+([.,!?;:])")     
        self.re_need_space_after   = re.compile(r"([.,!?;:])([^\s])")  

    def _normalize_unicode(self, s: str) -> str:
        s = unicodedata.normalize("NFC", s or "")
        for k, v in self.QUOTE_MAP.items():
            s = s.replace(k, v)
        return s

    def _strip_boilerplate(self, s: str) -> str:
        lines = []
        for line in s.splitlines():
            if any(rgx.search(line) for rgx in self.drop_res):
                continue
            lines.append(line)
        return "\n".join(lines)

    def clean_text(self, s: str) -> str:
        s = self._normalize_unicode(s)
        s = self._strip_boilerplate(s)

        s = self.re_ctrl.sub(" ", s)
        s = self.re_spaces.sub(" ", s)
        s = self.re_manynl.sub("\n\n", s)
        s = self.re_ws_nl.sub(lambda m: "\n", s)
        s = s.strip()

        s = s.replace("\n\n", f" {self.PARA} ")
        s = s.replace("\n", " ")

        s = self.re_space_before_punct.sub(r"\1", s)
        s = self.re_need_space_after.sub(r"\1 \2", s)

        s = re.sub(r"\s+", " ", s).strip()

        return s

    def process_series(self, texts: Union[pd.Series, Iterable[str]]) -> List[str]:
        seq = texts.tolist() if isinstance(texts, pd.Series) else list(texts)
        records = []
        for t in seq:
            clean = self.clean_text(str(t))
            if len(clean.split()) >= self.min_words:
                records.append(f"{self.BOS} {clean} {self.EOS}")
        return records


In [11]:
proc = TextPreprocessor(min_words=50)

train_records = proc.process_series(train_df["Texts"])
test_records = proc.process_series(test_df["Texts"])
val_records = proc.process_series(val_df["Texts"])

In [12]:
train_df.iloc[0]['Texts']

'Hòa Đa\nSanh Rớt\nThời gian gặp lại vợ con, sau khi học cải tạo cũng qua mau             Ngày tháng cứ dần trôi mà tôi không biết phải làm gì hơn là cứ quanh quẩn trông chừng hai đứa nhỏ, giúp vợ rảnh tay ngược xuôi, mua đầu chợ bán cuối chợ, kiếm sống qua ngày.  Cũng có lúc tôi thử bán buôn như vợ, nhưng có lẽ do bản tính thày giáo nên không thích ứng được với sinh hoạt chụp giựt, mua đấp bán đổi, lúc nào cũng phải nói láo để kiếm lời.  Hồng cũng cố gắng tạo điều kiện để tôi không phải làm những việc xem chừng không mấy thích hợp với tính khí của tôi... Rồi tôi cũng được một thằng bạn giúp tìm đồ nghề vá xe đạp cạnh lề đường; những lúc đó, Hồng ở nhà trông chừng con hay gửi con  hàng xóm, học trò cũ... để chạy hàng. Cuộc sống mới sao mà thê lương ảm đạm đến thế!             Một buổi chiều, Hồng về nhà trông có vẻ mệt mỏi và lo lắng: - Sáu Thủy mới gặp em nói, nhà mình có tên trong danh sách đi kinh tế mới, danh sách chỉ gồm những hộ ngụy quân ngụy quyền và mấy hộ có vấn đề về xã hội.

In [13]:
train_records[0]

'<BOS> Hòa Đa Sanh Rớt Thời gian gặp lại vợ con, sau khi học cải tạo cũng qua mau Ngày tháng cứ dần trôi mà tôi không biết phải làm gì hơn là cứ quanh quẩn trông chừng hai đứa nhỏ, giúp vợ rảnh tay ngược xuôi, mua đầu chợ bán cuối chợ, kiếm sống qua ngày. Cũng có lúc tôi thử bán buôn như vợ, nhưng có lẽ do bản tính thày giáo nên không thích ứng được với sinh hoạt chụp giựt, mua đấp bán đổi, lúc nào cũng phải nói láo để kiếm lời. Hồng cũng cố gắng tạo điều kiện để tôi không phải làm những việc xem chừng không mấy thích hợp với tính khí của tôi. .. Rồi tôi cũng được một thằng bạn giúp tìm đồ nghề vá xe đạp cạnh lề đường; những lúc đó, Hồng ở nhà trông chừng con hay gửi con hàng xóm, học trò cũ. .. để chạy hàng. Cuộc sống mới sao mà thê lương ảm đạm đến thế! Một buổi chiều, Hồng về nhà trông có vẻ mệt mỏi và lo lắng: - Sáu Thủy mới gặp em nói, nhà mình có tên trong danh sách đi kinh tế mới, danh sách chỉ gồm những hộ ngụy quân ngụy quyền và mấy hộ có vấn đề về xã hội. Ý này do bà Sáu Xê b

# Tokenize

In [14]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BOS, EOS, PAD, UNK, PARA = "<BOS>", "<EOS>", "<PAD>", "<UNK>", "<PARA>"

In [15]:
import re, unicodedata

class Tokenizer:
    def __init__(self, specials: List[str] = [BOS, EOS, PARA]):
        esc = [re.escape(s) for s in specials]
        specials_pat = "|".join(esc)
        self.tok_re = re.compile(
            rf"({specials_pat}|\.\.\.|…|[.,!?;:()\[\]\"'“”‘’—-]|[\w\-]+)",
            flags=re.UNICODE
        )

    def tokenize(self, s: str) -> List[str]:
        s = unicodedata.normalize("NFC", s or "")
        return [t for t in self.tok_re.findall(s) if t.strip()]

In [16]:
tokenizer = Tokenizer()
tokenizer.tokenize(train_records[0])[:50]

['<BOS>',
 'Hòa',
 'Đa',
 'Sanh',
 'Rớt',
 'Thời',
 'gian',
 'gặp',
 'lại',
 'vợ',
 'con',
 ',',
 'sau',
 'khi',
 'học',
 'cải',
 'tạo',
 'cũng',
 'qua',
 'mau',
 'Ngày',
 'tháng',
 'cứ',
 'dần',
 'trôi',
 'mà',
 'tôi',
 'không',
 'biết',
 'phải',
 'làm',
 'gì',
 'hơn',
 'là',
 'cứ',
 'quanh',
 'quẩn',
 'trông',
 'chừng',
 'hai',
 'đứa',
 'nhỏ',
 ',',
 'giúp',
 'vợ',
 'rảnh',
 'tay',
 'ngược',
 'xuôi',
 ',']

# Build Vocab

In [18]:
class Vocab:
    def __init__(self, min_freq = 2, max_size = 60000):
        self.min_freq = min_freq
        self.max_size = max_size
        self.stoi: Dict[str, int] = {}
        self.itos: Dict[int, str] = {}

    def build(self, texts: Iterable[str], tokenizer: Tokenizer):
        counter = Counter()
        for s in texts:
            counter.update(tokenizer.tokenize(s))

        # đảm bảo token đặc biệt lên đầu
        specials = [PAD, UNK, BOS, EOS, PARA]
        for sp in specials:
            counter[sp] += 10**9

        vocab = [w for w, c in counter.items() if c >= (0 if w in specials else self.min_freq)]
        vocab.sort(key=lambda w: (-counter[w], w))
        if self.max_size:
            vocab = vocab[:self.max_size]

        self.stoi = {w: i for i, w in enumerate(vocab)}
        self.itos = {i: w for w, i in self.stoi.items()}

    def __len__(self):
        return len(self.stoi)

    def encode(self, tokens: List[str]) -> List[int]:
        unk = self.stoi.get(UNK, None)
        return [self.stoi.get(t, unk) for t in tokens]

    def decode(self, ids: List[int]) -> List[str]:
        return [self.itos.get(i, UNK) for i in ids]

    def save(self, path: str):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"stoi": self.stoi, "itos": self.itos}, f, ensure_ascii=False)

    @staticmethod
    def load(path: str) -> "Vocab":
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        v = Vocab()
        v.stoi = {k: int(v_) if isinstance(v_, str) and v_.isdigit() else v_ for k, v_ in obj["stoi"].items()}
        v.itos = {int(k): v_ for k, v_ in obj["itos"].items()}
        return v

In [19]:
tokenizer = Tokenizer()

# build vocab
vocab = Vocab(max_size=50000, min_freq=1)
vocab.build(train_records, tokenizer)

In [43]:
print("Number of words in vocab:", len(vocab))
print("20 samples:", list(vocab.stoi.keys())[:20])

Number of words in vocab: 50000
20 samples: ['<BOS>', '<EOS>', '<PAD>', '<PARA>', '<UNK>', ',', '.', 'không', '-', 'một', 'có', 'là', 'của', ':', 'người', 'và', 'nói', 'đã', '?', 'lại', 'tôi', 'cho', 'ra', 'được', 'như', 'ta', 'đi', ';', 'trong', 'anh', '!', 'cũng', 'với', 'đến', 'mà', 'thì', 'phải', 'làm', 'những', 'vào', 'đó', 'con', '"', 'này', 'rồi', 'quot', 'lên', 'gì', 'để', 'ở']


In [22]:
tokens = tokenizer.tokenize("<BOS> Ronaldoooo Ronaldo ghi bàn thắng tiếp theo, đưa đội nhà dành chiến thắng ngược dòng. <EOS>")
ids = vocab.encode(tokens)
print(ids)
print(vocab.decode(ids))

[0, 4, 41060, 1278, 251, 871, 182, 118, 5, 179, 577, 70, 1217, 332, 871, 1114, 989, 6, 1]
['<BOS>', '<UNK>', 'Ronaldo', 'ghi', 'bàn', 'thắng', 'tiếp', 'theo', ',', 'đưa', 'đội', 'nhà', 'dành', 'chiến', 'thắng', 'ngược', 'dòng', '.', '<EOS>']


# Dataset and Dataloader

In [30]:
class LMDataset(Dataset):
    def __init__(self, records: List[str], tokenizer, vocab, max_len: int = 256, stride: Optional[int] = None, drop_last_short: bool = True):
        self.vocab = vocab
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.stride = stride or max_len
        self.samples: List[List[int]] = []

        for s in records:
            ids = vocab.encode(tokenizer.tokenize(s))
            # trượt cửa sổ theo stride
            for i in range(0, max(0, len(ids) - 1), self.stride):
                chunk = ids[i:i + max_len]
                if drop_last_short and len(chunk) < 4:
                    continue
                if len(chunk) >= 2:
                    self.samples.append(chunk)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx) -> Tuple[torch.Tensor, torch.Tensor]:
        seq = self.samples[idx]
        x = torch.tensor(seq[:-1], dtype=torch.long)
        y = torch.tensor(seq[1:],  dtype=torch.long)
        return x, y

def lm_collate(batch, pad_id: int):
    seqs, ys = zip(*batch)
    padded_x = pad_sequence(seqs, batch_first=True, padding_value=pad_id)
    padded_y = pad_sequence(ys,   batch_first=True, padding_value=pad_id)
    mask     = (padded_x != pad_id).long()
    return padded_x, padded_y, mask


In [53]:
pad_id = vocab.stoi["<PAD>"]
train_ds = LMDataset(train_records, tokenizer, vocab, max_len=128)
val_ds   = LMDataset(val_records,   tokenizer, vocab, max_len=128)
test_ds  = LMDataset(test_records,  tokenizer, vocab, max_len=128)

In [54]:
collate = partial(lm_collate, pad_id=pad_id)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, collate_fn=collate)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, collate_fn=collate)

xb, yb, mask = next(iter(train_loader))
print(xb.shape, yb.shape, mask.shape)

torch.Size([32, 127]) torch.Size([32, 127]) torch.Size([32, 127])


# Build Model

In [46]:
class LSTMLM(nn.Module):
    def __init__(self, vocab_size, emb_dim=256, hidden_dim=384, num_layers=2, pad_id=0, dropout=0.3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm  = nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers,
                             batch_first=True, dropout=dropout)
        self.drop  = nn.Dropout(dropout)
        self.fc    = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h=None):
        emb = self.embed(x)
        out, h = self.lstm(emb, h)
        out = self.drop(out)
        logits = self.fc(out)  # [B, T, V]
        return logits, h

In [47]:
pad_id = vocab.stoi["<PAD>"] 
model = LSTMLM(vocab_size=len(vocab), pad_id=pad_id)
print(model)

LSTMLM(
  (embed): Embedding(50000, 256, padding_idx=2)
  (lstm): LSTM(256, 384, num_layers=2, batch_first=True, dropout=0.3)
  (drop): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=384, out_features=50000, bias=True)
)


In [48]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params:,}")

Total params: 34,218,832


# Train

In [61]:
class LMTrainer:
    def __init__(
        self,
        model: nn.Module,
        pad_id: int,
        device: str = "cpu",
        lr: float = 3e-4,
        weight_decay: float = 0.01,
        use_scheduler: bool = True,
        amp: bool = True,             
        accum_steps: int = 1          
    ):
        self.model = model.to(device)
        self.pad_id = pad_id
        self.device = device
        self.amp = amp and torch.cuda.is_available()
        self.accum_steps = max(1, accum_steps)

        self.criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        self.scheduler = (
            torch.optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode="min", factor=0.5, patience=1)
            if use_scheduler else None
        )
        self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)

        self.history = {"train_loss": [], "val_loss": [], "train_ppl": [], "val_ppl": []}

    def _forward_loss(self, xb, yb):
        logits, _ = self.model(xb)  # [B,T,V]
        loss = self.criterion(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
        return loss, logits

    @torch.no_grad()
    def evaluate(self, loader):
        self.model.eval()
        total_loss, total_tok = 0.0, 0
        pbar = tqdm(loader, desc="Valid", leave=False)
        for xb, yb, mask in pbar:
            xb, yb, mask = xb.to(self.device), yb.to(self.device), mask.to(self.device)
            loss, _ = self._forward_loss(xb, yb)
            ntok = mask.sum().item()
            total_loss += loss.item() * ntok
            total_tok  += ntok
            avg_loss = total_loss / max(total_tok, 1)
            pbar.set_postfix(loss=f"{avg_loss:.4f}", ppl=f"{math.exp(avg_loss):.2f}")
        avg_loss = total_loss / max(total_tok, 1)
        ppl = math.exp(avg_loss)
        return avg_loss, ppl

    def train_one_epoch(self, loader, grad_clip=1.0, epoch_idx=1, epochs=1):
        self.model.train()
        total_loss, total_tok = 0.0, 0
        step = 0

        pbar = tqdm(loader, desc=f"Epoch {epoch_idx}/{epochs}", leave=True)
        autocast_ctx = torch.cuda.amp.autocast if self.amp else nullcontext

        self.optimizer.zero_grad(set_to_none=True)

        for xb, yb, mask in pbar:
            xb, yb, mask = xb.to(self.device), yb.to(self.device), mask.to(self.device)
            with autocast_ctx():
                loss, _ = self._forward_loss(xb, yb)
                loss_to_backprop = loss / self.accum_steps

            # backward (AMP)
            if self.amp:
                self.scaler.scale(loss_to_backprop).backward()
            else:
                loss_to_backprop.backward()

            step += 1
            do_step = (step % self.accum_steps == 0)

            if do_step:
                if grad_clip is not None:
                    if self.amp:
                        self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), grad_clip)

                if self.amp:
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    self.optimizer.step()

                self.optimizer.zero_grad(set_to_none=True)

            # logging tiến trình
            ntok = mask.sum().item()
            total_loss += loss.item() * ntok
            total_tok  += ntok
            avg_loss = total_loss / max(total_tok, 1)
            ppl = math.exp(avg_loss)

            # thông tin thêm: LR + (tuỳ) bộ nhớ GPU
            lr = self.optimizer.param_groups[0]["lr"]
            postfix = dict(loss=f"{avg_loss:.4f}", ppl=f"{ppl:.2f}", lr=f"{lr:.2e}")
            if torch.cuda.is_available() and self.device.startswith("cuda"):
                free, total = torch.cuda.mem_get_info()
                used_gb = (total - free) / (1024**3)
                postfix["gpu_used"] = f"{used_gb:.1f}G"
            pbar.set_postfix(postfix)

        avg_loss = total_loss / max(total_tok, 1)
        ppl = math.exp(avg_loss)
        return avg_loss, ppl

    def fit(self, train_loader, val_loader, epochs=6, ckpt_path="lstm_lm.pt", grad_clip=1.0, patience=2):
        best_val = float("inf")
        bad_epochs = 0

        for ep in range(1, epochs + 1):
            tr_loss, tr_ppl = self.train_one_epoch(train_loader, grad_clip=grad_clip, epoch_idx=ep, epochs=epochs)
            va_loss, va_ppl = self.evaluate(val_loader)

            self.history["train_loss"].append(tr_loss)
            self.history["val_loss"].append(va_loss)
            self.history["train_ppl"].append(tr_ppl)
            self.history["val_ppl"].append(va_ppl)

            print(f"Epoch {ep:02d} | Train PPL: {tr_ppl:.2f} | Val PPL: {va_ppl:.2f}")

            if self.scheduler:
                self.scheduler.step(va_loss)

            if va_ppl < best_val:
                best_val = va_ppl
                torch.save({"model": self.model.state_dict()}, ckpt_path)
                bad_epochs = 0
                print("  -> Saved best model")
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print(f"Early stopping at epoch {ep} (no improvement for {patience} epochs).")
                    break

    def test(self, test_loader, ckpt_path="lstm_lm.pt"):
        ckpt = torch.load(ckpt_path, map_location=self.device)
        self.model.load_state_dict(ckpt["model"])
        test_loss, test_ppl = self.evaluate(test_loader)
        print(f"Test PPL: {test_ppl:.2f}")
        return test_loss, test_ppl

In [56]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [62]:
pad_id = vocab.stoi["<PAD>"]

model = LSTMLM(vocab_size=len(vocab), emb_dim=256, hidden_dim=384, num_layers=2, pad_id=pad_id, dropout=0.3)

trainer = LMTrainer(
    model, pad_id=pad_id, device=DEVICE,
    lr=3e-4, weight_decay=0.01,
    amp=True,          
    accum_steps=2        
)

trainer.fit(train_loader, val_loader, epochs=2, ckpt_path="lstm_lm.pt", grad_clip=1.0, patience=2)
trainer.test(test_loader, ckpt_path="lstm_lm.pt")


/tmp/ipykernel_37/1425706927.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)
Epoch 1/5:   0%|          | 0/57028 [00:00<?, ?it/s]/tmp/ipykernel_37/1425706927.py:63: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast_ctx():
Epoch 1/5:   0%|          | 124/57028 [00:13<1:46:22,  8.92it/s, loss=8.3792, ppl=4355.38, lr=3.00e-04, gpu_used=14.8G] 


KeyboardInterrupt: 